In [1]:
import json
from glob import glob
from tqdm import tqdm
import html
import re
import os

os.makedirs("Postprocessed Outputs", exist_ok=True)

In [2]:
def postprocess(text):
    # check text is string, say if None or other type
    if not isinstance(text, str):
        return "NA"
    
    text = text.strip()
    # supernote (Repeatedly remove matching wrapping quotes """<FULL_COMM_NOTE_INSIDE>""")
    while (len(text) >= 2 and text[0] == text[-1] and text[0] in ("'", '"')):
        text = text[1:-1].strip()
    # human (&quot; --> ", &amp; --> & etc.)
    text = html.unescape(text)
    # openai's gpt (?utm_source=openai removed to reduce source bias)
    text = re.sub(r'\?utm_source=openai','',text)

    return text

In [3]:
files = sorted(glob('Outputs/*.json'))

for f in tqdm(files):
    dic = json.load(open(f, encoding='utf-8'))

    dic = {k: postprocess(v) for k, v in dic.items()}
    json.dump(dic, open(f"Postprocessed Outputs/{os.path.basename(f)}", "w", encoding='utf-8'), indent=4)

100%|██████████| 15/15 [00:00<00:00, 305.97it/s]
